# IKEA Product Price Analysis & Machine Learning

This notebook presents an end-to-end analysis of IKEA products using Python.

The project covers:

- data loading and cleaning;
- exploratory data analysis;
- feature engineering;
- statistical hypothesis testing;
- correlation and regression analysis;
- comparison of machine learning models;
- hyperparameter tuning;
- model evaluation;
- feature importance;
- cross-validation.

The goal is to investigate which product characteristics are associated with IKEA product prices and build a model for price prediction.


In [ ]:
# Core libraries
import warnings
warnings.filterwarnings("ignore")

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from scipy.stats import (
    ttest_ind,
    mannwhitneyu,
    pearsonr,
    spearmanr
)

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")


In [ ]:
# Load data

URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-11-03/ikea.csv"

def download_dataset(url: str) -> pd.DataFrame:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return pd.read_csv(url, index_col=0)

df_raw = download_dataset(URL)

print(f"Dataset shape: {df_raw.shape}")
df_raw.head()


## 1. Initial Data Inspection

In [ ]:
df_raw.info()


In [ ]:
df_raw.describe(include="all").T


In [ ]:
missing = (
    df_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

missing["missing_pct"] = (
    missing["missing_values"] / len(df_raw) * 100
).round(2)

missing[missing["missing_values"] > 0]


In [ ]:
print("Duplicate item IDs:", df_raw["item_id"].duplicated().sum())
print("Duplicate rows:", df_raw.duplicated().sum())


## 2. Data Cleaning & Feature Engineering

In [ ]:
df = df_raw.copy()

# Keep one record per product
df = df.drop_duplicates(subset="item_id").copy()

# Standardize the additional-colors field
df["other_colors"] = (
    df["other_colors"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df = df[df["other_colors"].isin(["yes", "no"])].copy()

# Convert dimensions to numeric and impute missing values with medians
dimension_cols = ["depth", "height", "width"]

for col in dimension_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Remove implausible dimension values
df = df[
    (df["height"] < 300) &
    (df["width"] < 300) &
    (df["depth"] < 300)
].copy()

# Product volume in cubic meters
df["size"] = (
    df["depth"] * df["height"] * df["width"] / 1_000_000
)

# Clean designer
df["designer_new"] = df["designer"].astype("string")

df.loc[
    df["designer_new"].str.contains(r"\d", na=False),
    "designer_new"
] = pd.NA

df["designer_new"] = (
    df["designer_new"]
    .str.split("/")
    .str[0]
    .str.strip()
    .replace("", pd.NA)
    .fillna("IKEA of Sweden")
)

# Clean old price
df["old_price_new"] = (
    df["old_price"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
)

df["old_price_new"] = pd.to_numeric(
    df["old_price_new"], errors="coerce"
)

df["old_price_new"] = df["old_price_new"].fillna(df["price"])

# Clean product/series name
df["name_new"] = (
    df["name"]
    .astype("string")
    .str.split("/")
    .str[0]
    .str.strip()
    .replace("", pd.NA)
)

# Log-transformed price for distributional analysis
df["price_ln"] = np.log(df["price"].clip(lower=1))

# Human-readable online availability
df["sellable_online_str"] = df["sellable_online"].map({
    True: "Online",
    False: "Offline"
})

print(f"Rows after cleaning: {len(df):,}")
print(f"Columns: {df.shape[1]}")


In [ ]:
df.head()


In [ ]:
df[dimension_cols + ["size", "price", "old_price_new"]].describe().T


## 3. Exploratory Data Analysis

In [ ]:
# Product categories

category_counts = df["category"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
ax = category_counts.plot(
    kind="bar",
    color=sns.color_palette("viridis", len(category_counts))
)
ax.bar_label(ax.containers[0], fontsize=9)

plt.title("Number of Products by Category", fontweight="bold")
plt.xlabel("Category")
plt.ylabel("Number of Products")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Price distribution

plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=30, kde=True)

plt.title("Distribution of IKEA Product Prices", fontweight="bold")
plt.xlabel("Price")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()


In [ ]:
# Log-transformed price distribution

plt.figure(figsize=(10, 5))
sns.histplot(df["price_ln"], bins=30, kde=True)

plt.title("Distribution of Log-Transformed Prices", fontweight="bold")
plt.xlabel("Log(price)")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()


In [ ]:
# Old vs current price

plt.figure(figsize=(9, 6))
sns.regplot(
    data=df,
    x="old_price_new",
    y="price",
    scatter_kws={"alpha": 0.5},
    line_kws={"color": "red"}
)

plt.title("Old Price vs Current Price", fontweight="bold")
plt.xlabel("Old Price")
plt.ylabel("Current Price")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation between price and dimensions

corr = df[["price", "depth", "height", "width"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    square=True
)

plt.title("Correlation Between Price and Product Dimensions", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Top designers by number of products

top_designers = (
    df["designer_new"]
    .value_counts()
    .head(15)
    .sort_values()
)

plt.figure(figsize=(10, 7))
ax = top_designers.plot(kind="barh")

ax.bar_label(ax.containers[0], fontsize=9)

plt.title("Top 15 Designers by Number of Products", fontweight="bold")
plt.xlabel("Number of Products")
plt.ylabel("Designer")
plt.tight_layout()
plt.show()


In [ ]:
# Top product series by median price

top_series_price = (
    df.groupby("name_new")["price"]
    .median()
    .dropna()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)

plt.figure(figsize=(10, 6))
ax = top_series_price.plot(kind="barh")

ax.bar_label(
    ax.containers[0],
    labels=[f"${x:.0f}" for x in top_series_price],
    fontsize=9
)

plt.title("Top 10 Product Series by Median Price", fontweight="bold")
plt.xlabel("Median Price")
plt.ylabel("Product Series")
plt.tight_layout()
plt.show()


In [ ]:
# Additional colors: product count and median price

color_summary = (
    df.groupby("other_colors")
    .agg(
        product_count=("price", "size"),
        median_price=("price", "median")
    )
    .reindex(["no", "yes"])
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(
    ["No additional colors", "Additional colors"],
    color_summary["product_count"]
)
axes[0].set_title("Products by Color Availability", fontweight="bold")
axes[0].set_ylabel("Number of Products")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(
    ["No additional colors", "Additional colors"],
    color_summary["median_price"]
)
axes[1].set_title("Median Price by Color Availability", fontweight="bold")
axes[1].set_ylabel("Median Price")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

color_summary


In [ ]:
# Online vs offline products

online_summary = (
    df.groupby("sellable_online")
    .agg(
        product_count=("price", "size"),
        median_price=("price", "median")
    )
    .reindex([False, True])
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(
    ["Offline", "Online"],
    online_summary["product_count"]
)
axes[0].set_title("Online vs Offline Products", fontweight="bold")
axes[0].set_ylabel("Number of Products")

axes[1].bar(
    ["Offline", "Online"],
    online_summary["median_price"]
)
axes[1].set_title("Median Price: Online vs Offline", fontweight="bold")
axes[1].set_ylabel("Median Price")

plt.tight_layout()
plt.show()

online_summary


## 4. Statistical Hypothesis Testing

### Hypothesis 1 — Additional Colors and Price

**H₀:** The price distributions are the same for products with and without additional color options.

**H₁:** The price distributions differ.

The Mann–Whitney U test is used as the primary non-parametric test. An independent samples t-test is included as a complementary analysis.

A bootstrap confidence interval is calculated for the difference in mean prices.


In [ ]:
with_colors = df.loc[
    df["other_colors"] == "yes", "price"
].dropna()

without_colors = df.loc[
    df["other_colors"] == "no", "price"
].dropna()

median_with_colors = with_colors.median()
median_without_colors = without_colors.median()

mw_stat, mw_p = mannwhitneyu(
    with_colors,
    without_colors,
    alternative="two-sided"
)

t_stat, t_p = ttest_ind(
    with_colors,
    without_colors,
    equal_var=False
)

print(f"Median price — with additional colors: ${median_with_colors:.2f}")
print(f"Median price — without additional colors: ${median_without_colors:.2f}")
print(f"Median difference: ${median_with_colors - median_without_colors:.2f}")
print(f"Mann–Whitney U p-value: {mw_p:.6g}")
print(f"Welch's t-test p-value: {t_p:.6g}")


In [ ]:
# Bootstrap confidence interval for the difference in means

def bootstrap_mean_difference(
    data1,
    data2,
    n_iterations=5000,
    random_state=RANDOM_STATE
):
    rng = np.random.default_rng(random_state)
    differences = np.empty(n_iterations)

    for i in range(n_iterations):
        sample1 = rng.choice(
            data1,
            size=len(data1),
            replace=True
        )
        sample2 = rng.choice(
            data2,
            size=len(data2),
            replace=True
        )
        differences[i] = sample1.mean() - sample2.mean()

    return differences

bootstrap_diffs = bootstrap_mean_difference(
    with_colors.to_numpy(),
    without_colors.to_numpy()
)

ci_colors = np.percentile(
    bootstrap_diffs,
    [2.5, 97.5]
)

print(
    "95% bootstrap CI for mean difference: "
    f"[${ci_colors[0]:.2f}, ${ci_colors[1]:.2f}]"
)

plt.figure(figsize=(9, 5))
sns.histplot(bootstrap_diffs, bins=40, kde=True)

plt.axvline(ci_colors[0], linestyle="--", label="2.5%")
plt.axvline(ci_colors[1], linestyle="--", label="97.5%")
plt.axvline(0, linestyle="-", label="No difference")

plt.title("Bootstrap Distribution of Mean Price Difference", fontweight="bold")
plt.xlabel("Mean price difference: with colors − without colors")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


### Hypothesis 2 — Product Size and Price

**H₀:** Product size is not associated with price.

**H₁:** Product size is associated with price.

Pearson and Spearman correlations are used to evaluate the relationship from both linear and rank-based perspectives.


In [ ]:
pearson_corr, pearson_p = pearsonr(
    df["size"],
    df["price"]
)

spearman_corr, spearman_p = spearmanr(
    df["size"],
    df["price"]
)

print(f"Pearson correlation: {pearson_corr:.4f}")
print(f"Pearson p-value: {pearson_p:.6g}")
print(f"Spearman correlation: {spearman_corr:.4f}")
print(f"Spearman p-value: {spearman_p:.6g}")


In [ ]:
# OLS regression: price ~ size

X_ols = sm.add_constant(df["size"])
ols_model = sm.OLS(df["price"], X_ols).fit()

print(ols_model.summary())


In [ ]:
# Bootstrap confidence interval for the correlation

rng = np.random.default_rng(RANDOM_STATE)
boot_correlations = np.empty(5000)

for i in range(5000):
    sample = df.sample(
        n=len(df),
        replace=True,
        random_state=int(rng.integers(0, 1_000_000_000))
    )
    boot_correlations[i] = sample["size"].corr(sample["price"])

ci_corr = np.percentile(
    boot_correlations,
    [2.5, 97.5]
)

print(
    "95% bootstrap CI for Pearson-like sample correlation: "
    f"[{ci_corr[0]:.4f}, {ci_corr[1]:.4f}]"
)

plt.figure(figsize=(9, 5))
plt.hist(boot_correlations, bins=40)

plt.axvline(ci_corr[0], linestyle="--", label="2.5%")
plt.axvline(ci_corr[1], linestyle="--", label="97.5%")
plt.axvline(
    boot_correlations.mean(),
    linestyle="-",
    label="Mean bootstrap correlation"
)

plt.title("Bootstrap Distribution of Size–Price Correlation", fontweight="bold")
plt.xlabel("Correlation")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Size vs price

plt.figure(figsize=(9, 6))
sns.regplot(
    data=df,
    x="size",
    y="price",
    scatter_kws={"alpha": 0.4},
    line_kws={"color": "red"}
)

plt.title("Relationship Between Product Size and Price", fontweight="bold")
plt.xlabel("Product Size (m³)")
plt.ylabel("Price")
plt.tight_layout()
plt.show()


## 5. Machine Learning — Price Prediction

### Objective

Build regression models that predict IKEA product prices from product characteristics.

**Target:** `price`

To avoid target leakage, all features are created independently of the target variable. In particular, category-level and designer-level median prices based on `price` are **not used as model features**.


In [ ]:
# Select features without target-derived information

target = "price"

feature_columns = [
    "height",
    "width",
    "depth",
    "name_new",
    "other_colors"
]

X = df[feature_columns].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")


In [ ]:
numeric_features = [
    "height",
    "width",
    "depth"
]

categorical_features = [
    "name_new",
    "other_colors"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5
        )
    )
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


## 6. Model Comparison

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE
    )
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

model_results = []

for name, estimator in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="r2",
        n_jobs=-1
    )

    model_results.append({
        "model": name,
        "mean_r2": scores.mean(),
        "std_r2": scores.std()
    })

model_results = (
    pd.DataFrame(model_results)
    .sort_values("mean_r2", ascending=False)
    .reset_index(drop=True)
)

model_results


In [ ]:
plt.figure(figsize=(8, 5))

ax = plt.bar(
    model_results["model"],
    model_results["mean_r2"]
)

plt.title("Model Comparison — 5-Fold Cross-Validation", fontweight="bold")
plt.ylabel("Mean R²")
plt.xlabel("Model")
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()


## 7. Random Forest Hyperparameter Tuning

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 30],
    "model__min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print(f"Best CV R²: {grid_search.best_score_:.4f}")


## 8. Final Model Evaluation

In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

test_r2 = r2_score(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

evaluation = pd.DataFrame({
    "Metric": ["R²", "MAE", "RMSE"],
    "Value": [test_r2, test_mae, test_rmse]
})

evaluation


In [ ]:
# Actual vs predicted prices

plt.figure(figsize=(8, 6))
plt.scatter(
    y_test,
    y_pred,
    alpha=0.5
)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.title("Actual vs Predicted IKEA Product Prices", fontweight="bold")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.tight_layout()
plt.show()


## 9. Feature Importance

In [ ]:
# Extract transformed feature names and Random Forest importances

fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_forest = best_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_forest.feature_importances_

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

feature_importance.head(20)


In [ ]:
# Aggregate one-hot encoded features into meaningful feature groups

def feature_group(feature_name: str) -> str:
    if "num__width" in feature_name:
        return "Width"
    if "num__height" in feature_name:
        return "Height"
    if "num__depth" in feature_name:
        return "Depth"
    if "cat__name_new" in feature_name:
        return "Product series"
    if "cat__other_colors" in feature_name:
        return "Additional colors"
    return "Other"

feature_importance["group"] = (
    feature_importance["feature"]
    .map(feature_group)
)

grouped_importance = (
    feature_importance
    .groupby("group")["importance"]
    .sum()
    .sort_values(ascending=False)
)

grouped_importance_pct = (
    grouped_importance /
    grouped_importance.sum() * 100
)

grouped_importance_pct


In [ ]:
plt.figure(figsize=(9, 5))

sorted_importance = grouped_importance_pct.sort_values()

bars = plt.barh(
    sorted_importance.index,
    sorted_importance.values
)

for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}%",
        va="center"
    )

plt.title("Aggregated Random Forest Feature Importance", fontweight="bold")
plt.xlabel("Importance (%)")
plt.ylabel("Feature Group")
plt.tight_layout()
plt.show()


## 10. Final Cross-Validation

In [ ]:
# Repeated evaluation of the complete best pipeline on the training data.
# The test set remains untouched for the final performance estimate.

final_cv_scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print("Fold R² scores:")
for i, score in enumerate(final_cv_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean CV R²: {final_cv_scores.mean():.4f}")
print(f"CV R² Std: {final_cv_scores.std():.4f}")


In [ ]:
plt.figure(figsize=(8, 5))

folds = np.arange(1, len(final_cv_scores) + 1)

plt.plot(
    folds,
    final_cv_scores,
    marker="o"
)

for fold, score in zip(folds, final_cv_scores):
    plt.text(
        fold,
        score,
        f"{score:.2f}",
        ha="center",
        va="bottom"
    )

plt.xticks(folds)
plt.title("Random Forest Cross-Validation Scores", fontweight="bold")
plt.xlabel("Fold")
plt.ylabel("R²")
plt.tight_layout()
plt.show()


## 11. Final Summary

In [ ]:
summary = {
    "Original rows": len(df_raw),
    "Cleaned rows": len(df),
    "Pearson correlation (size vs price)": round(pearson_corr, 4),
    "Spearman correlation (size vs price)": round(spearman_corr, 4),
    "Mann–Whitney p-value (colors vs price)": mw_p,
    "Best CV R²": round(grid_search.best_score_, 4),
    "Test R²": round(test_r2, 4),
    "Test MAE": round(test_mae, 4),
    "Test RMSE": round(test_rmse, 4),
    "Final CV mean R²": round(final_cv_scores.mean(), 4),
    "Final CV std": round(final_cv_scores.std(), 4)
}

pd.Series(summary)


## Conclusions

The analysis combines exploratory analysis, statistical testing and machine learning to investigate IKEA product pricing.

Key conclusions should be based on the values produced above after running the notebook:

1. Product size shows a measurable association with price.
2. Products with additional color options can be statistically compared with products without them; the significance and effect size are reported above.
3. Random Forest is evaluated against Linear Regression and Gradient Boosting using the same cross-validation procedure.
4. Hyperparameter tuning is performed without exposing the test set to model selection.
5. The final model is evaluated on a completely held-out test set.
6. Feature importance is interpreted as model importance, not as causal influence.

### Reproducibility

All random procedures use `random_state=42`, and preprocessing is contained inside Scikit-learn pipelines to prevent information leakage between training and validation/test data.
